In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# import crest
%cd -q ../..
from crest import HierarchalTensorGraph

## HTG basenodes

There are 2 modes distinguished in HTGs. The first is a graph containing only one node and referred to
as a basenode. Basenodes represent a fundamental physical process within the Earth System. Basenodes are
created by passing a callable for the 'node' value in HTG __init__ when node (basenode).

In [ ]:
# A simple basenode that squares a number
process = lambda X : {'square' : X['x']*X['x']}
sq = HierarchalTensorGraph(node=process)

In [ ]:
# All HTGs must have a name. If no name is specified,
# HTG will try standard naming variables
sq.name

In [ ]:
# Best to always name them if what you want during construction
# sq = HierarchalTensorGraph(node=process,name='square') 
# or simply
sq.name = 'square'

In [ ]:
# Every node contains a graph object the includes the nodes and
# edges within it. A basenode has an 'empty' graph.
# By design, you can never add nodes to a basenode.
if sq.is_empty:
    print(f'{sq.name} has an empty graph!')

In [ ]:
# Let's execute the HTG by creating the input dict
X = {'x' : 2}
sq(X)

## Multi-node HTGs

Mutlinode HTG are used to represent coupled systems. Nodes in this mode can either
be basenodes or other multinode HTGs.

In [ ]:
# Let's now create a HTG with multiple nodes to create a graph for a*(b_c)
process = lambda X : {'sum' : X['x_1'] + X['x_2']}     
add = HierarchalTensorGraph(node=process,name='add')    
add_square = HierarchalTensorGraph(name='add_square') # Only the name needs to be given

# Specify the graph
add_square.add_edge('input',add)                       
add_square.add_edge(sq,'output')
add_square.add_edge(add,sq)
add_square.draw()

In [ ]:
# Accessing nodes with __getitem__
f'__getitem__ = {add_square["add"]}'

In [ ]:
# Get a dictionary of the nodes in the HTG
add_square.nodes

## Renaming I/O in HTGs 

It will often be necessary to rename I/O in the graphs to match
the naming conventions used by the nodes in your graph.

In [ ]:
# It's best practice to set the the names of input/output tensors.
# A HTG will use this information to try and find and select the input
# it needs. Note: you can do this at construction time as well by passing
# inputs={} and outputs={}
add_square['add'].inputs = {'x_1' : None, 'x_2' : None}
add_square['add'].outputs= {'sum' : None} #leave the output as y
add_square['square'].inputs = {'x' : None}
add_square['square'].outputs = {'square' : None} #leave the output as y

In [ ]:
# We have to rename the output from the addition node
# from y to x in order to match the input of the square node
# You can rename any incoming inputs or outgoing outputs
# using renami_io.
add_square.rename_io(outputs_map={'sum' : 'x'}, node='add')

In [ ]:
X = {'x_1' : 5, 'x_2' : 3}
add_square(X)

In [ ]:
# Notice we get a nested dict for output because
# HTG prepends the nodename to ensure unique names
# in the graph. To fix this, as recommended, we simply
# set the inputs/outputs of add_square.
add_square.inputs = {'x_1' : None, 'x_2' : None}
add_square.outputs = {'square' : None} # this is the only needed to fix the output

In [ ]:
# Let's check that other input keys are ignored.
X = {'x_1' : 5, 'x_2' : 3, 'z' : 12}
add_square(X)

In [ ]:
# Sometimes we may want to wrap our HTG in a class for more flexibility
class AddSquare(HierarchalTensorGraph):
    
    def __init__(self,name='add_square'):
        # Construct a multinode HTG and specify I/O
        super().__init__(name=name,
                         inputs = {'x_1' : None, 'x_2' : None},
                         outputs = {'add_square' : None}
        )
        
        # Create an add basenode and specify I/O
        add = HierarchalTensorGraph(
            lambda X : {'sum' : X['x_1'] + X['x_2']},
            inputs = {'x_1' : None, 'x_2' : None},
            outputs = {'sum' : None},
            name = 'add'
        )
        
        # Create a square basenode and specify I/O
        square = HierarchalTensorGraph(
            lambda X : {'square' : X['x']*X['x']},
            inputs = {'x' : None},
            outputs = {'square' : None},
            name = 'square'
        )
            
        # Create the graph    
        self.add_edge('input',add)
        self.add_edge(add,square)
        self.add_edge(square,'output')
        
        # Don't forget to rename the output of add
        self.rename_io(outputs_map={'sum' : 'x'}, node='add')
        
        # And of square since we specified add_square for the output
        self.rename_io(outputs_map={'square' : 'add_square'}, node=square)

In [ ]:
# Evaluate our class
X = {'x_1' : 5, 'x_2' : 3, 'z' : 12}
add_sq = AddSquare()
add_sq(X)

## An HTG with one more layer

Here, we will add another layer to do log(a*(b+c))

In [ ]:
#et's create another layer of the hierarchy.
from math import log

class LogAddSquare(HierarchalTensorGraph):
        
    def __init__(self,name='log_add_square'):
        super().__init__(
            name=name,
            inputs = {'x_1' : None, 'x_2' : None},
            outputs = {name : None}
        )
        
        # Create log basenode
        lg = HierarchalTensorGraph(
            name = 'log',
            node = lambda X : {'log' : log(X['x'])},
            inputs = {'x' : None},
            outputs = {'log' : None}
        )
        
        # Create graph
        self.add_edge('input',AddSquare())
        self.add_edge('add_square',lg)
        self.add_edge('log','output')
        self.add_edge('add_square','output')
        # Rename log add_square output to match log input
        self.rename_io(outputs_map={'add_square' : 'x'}, node='add_square')

        # Rename log output to match output of LogAddSquare
        self.rename_io(outputs_map={'log' : 'log_add_square'}, node='log')

In [ ]:
# Execute the HTG
las = LogAddSquare()
las(X)

In [ ]:
# Draw the HTG
las.draw()

In [ ]:
# Draw the HTG with add_square 'expanded'
las.draw(expand_nodes='add_square')

In [ ]:
# List its nodes
las.nodes

In [ ]:
# List ALL nodes within the whole system
# Note: tuples are used to specify nodes uniquely
# that are deeper in the graph
las.all_nodes

In [ ]:
# We can access nodes of the parent simply with their names
las['log']

In [ ]:
# We could also give the name as a tuple
las[('log')]

In [ ]:
# If we want to access nodesdeeper within the HTG, we MUST use a tuple
las[('add_square','add')]

In [ ]:
# Iterate through ALL nodes in a HTG
for i in las:
    print(i)

In [ ]:
# Ask if a node is in a HTG
print([
    'log' in las, 
    ('log') in las, 
    ('add_square','add') in las,
    'add' in las]
)

In [ ]:
# Access to individual node output. Every node stores
# the output of the most recent call which can be accessed as
print([las['log'].output,las[('add_square','add')].output])

In [ ]:
# store the graph in a json file
import json 

with open('crest_HTG_overview.json', 'w') as outfile:
    json_data = las.to_json()
    json.dump(json_data, outfile)

In [ ]:
# load the graph from a json file

sample = lambda X : X
sam = HierarchalTensorGraph(node=sample)

with open('crest_HTG_overview.json', 'r') as infile:
    y = json.load(infile)
    sam.from_json(y)

for i in sam:
    print(i)